<a href="https://colab.research.google.com/github/adhamov8/BDA_HW-shopper-analytics/blob/main/BDA_HW3_KMeans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shopper · HW3 3: K-means and cluster interpretation


Alibek Adhamov · Basic Methods of Data Analytics

Which browsing patterns recur across 500 online shop sessions, and how much detail is useful to describe them? Compare K = 4 and K = 7 on identical inputs. Data: [Online Shoppers Purchasing Intention, UCI](https://doi.org/10.24432/C5F88Q), CC BY 4.0; the repository CSV is the fixed sample (seed 42) used in HW1 and HW2.

## A3.1 · Select features, standardize and compare random starts
Use page counts and durations in three sections: administrative, informational and product-related. Counts describe breadth; duration describes time spent. Standardize each column using its sample-wide mean and population standard deviation so that seconds do not dominate counts simply because of their units. Neither purchase outcome nor source ID contributes to distances.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from threadpoolctl import threadpool_limits

name = 'shopper_sample_500_seed42.csv'
url = 'https://raw.githubusercontent.com/adhamov8/BDA_HW-shopper-analytics/main/' + name
df = pd.read_csv(name if Path(name).exists() else url)
features = ['Administrative','Administrative_Duration','Informational',
            'Informational_Duration','ProductRelated','ProductRelated_Duration']
X = df[features]
assert X.shape == (500, 6) and X.notna().all().all()
Z = StandardScaler().fit_transform(X)
print('Sessions × features:', X.shape)

Sessions × features: (500, 6)


Each run starts with randomly selected observations and minimizes the sum of squared distances to the assigned centers (inertia). Record all 15 starts per K. Select the smallest observed inertia separately for each K; this does not guarantee a global optimum.

In [ ]:
best, records = {}, []
with threadpool_limits(limits=1):
    for k in (4, 7):
        models = []
        for seed in range(15):
            model = KMeans(n_clusters=k, init='random', n_init=1, random_state=seed,
                           algorithm='lloyd', max_iter=300, tol=1e-4).fit(Z)
            models.append(model)
            records.append((k, seed, model.inertia_, model.n_iter_))
        best[k] = min(models, key=lambda m: m.inertia_)
runs = pd.DataFrame(records, columns=['K','seed','inertia','iterations'])
print(runs.pivot(index='seed', columns='K', values=['inertia','iterations']).round(3))
for k, model in best.items():
    print(f'K={k}: selected seed {model.random_state}, inertia {model.inertia_:.3f}')

       inertia           iterations      
K            4         7          4     7
seed                                     
0     1539.761   886.042       21.0  29.0
1     1383.268  1048.543       15.0  18.0
2     1539.752   872.840       19.0  23.0
3     1365.752   991.763        9.0  27.0
4     1426.695  1050.396        9.0   8.0
5     1383.268   991.763       24.0  42.0
6     1369.723  1016.673       25.0  15.0
7     1383.268  1013.077       18.0  19.0
8     1537.286   872.406        9.0  24.0
9     1539.752   872.406       15.0  27.0
10    1383.268   872.406       22.0  16.0
11    1383.268   887.134       21.0  24.0
12    1539.752  1128.251       17.0  23.0
13    1391.762   885.881       11.0  12.0
14    1383.268   885.881       20.0  16.0
K=4: selected seed 3, inertia 1365.752
K=7: selected seed 8, inertia 872.406


## A3.2 · Describe both partitions
Cluster centers below are means in original units. Relative difference = 100 × (cluster mean − grand mean) / grand mean. Positive values indicate above-average activity. IDs in the tables equal the fitted label plus one. A large percentage can result when the grand mean is small: read it alongside the original center and the number of sessions.

In [ ]:
X = X.set_axis(['Admin_n','Admin_sec','Info_n','Info_sec','Product_n','Product_sec'],axis=1)
grand = X.mean()
print('Grand mean in original units:')
print(grand.round(2).to_string())
for k,m in best.items():
    grouped = X.groupby(m.labels_+1)
    centers = grouped.mean()
    sizes = grouped.size().rename('sessions')
    deviation = 100*(centers/grand - 1)
    assert sizes.sum()==500 and np.allclose(np.average(centers,axis=0,weights=sizes),grand)
    print(f'\nK={k} | centers in original units')
    print(pd.concat([sizes,centers],axis=1).round(2).to_string())
    print('Relative differences from the grand mean (%)')
    print(deviation.round(1).to_string())

Grand mean in original units:
Admin_n           2.54
Admin_sec        86.71
Info_n            0.48
Info_sec         32.12
Product_n        32.68
Product_sec    1229.00

K=4 | centers in original units
   sessions  Admin_n  Admin_sec  Info_n  Info_sec  Product_n  Product_sec
1       119     5.55     199.75    0.39     13.64      43.46      1514.15
2        16    10.56     388.07    1.69     72.06     213.88      8193.64
3        33     7.48     258.00    4.00    370.42      65.00      2682.29
4       332     0.58      14.65    0.11      3.19      16.87       646.70
Relative differences from the grand mean (%)
   Admin_n  Admin_sec  Info_n  Info_sec  Product_n  Product_sec
1    118.7      130.4   -19.5     -57.5       33.0         23.2
2    316.5      347.5   251.6     124.4      554.5        566.7
3    195.1      197.5   733.3    1053.4       98.9        118.2
4    -77.2      -83.1   -78.0     -90.1      -48.4        -47.4

K=7 | centers in original units
   sessions  Admin_n  Admin_sec

### Four groups
|ID|Sessions|Profile relative to the grand mean|
|---|---:|---|
|1|119|Administrative browsing: administrative page count and time are about 119% and 130% above average. Informational time is 58% below average.|
|2|16|Extensive product browsing: every mean is elevated, especially product pages (+554%) and product time (+567%).|
|3|33|Information-intensive: informational page count is 733% above average and informational time is 1,053% above average.|
|4|332|Light browsing: all six means are below average, including product pages (−48%) and administrative time (−83%).|

### Seven groups
|ID|Sessions|Profile relative to the grand mean|
|---|---:|---|
|1|4|Extreme product browsing: product count is 1,067% above average and product time is 1,222% above average. Only four sessions support this profile.|
|2|49|Product-focused: product time is 153% above average while administrative and informational time are both below average.|
|3|32|Long administrative visits: all six means are elevated; administrative time shows the greatest excess (+505%).|
|4|276|Light across all sections: product time is 67% below average and informational time is 94% below average.|
|5|10|Very long informational visits: informational time is 2,197% above average. Product time is also elevated (+357%).|
|6|32|Information-focused: informational page count and time are high (+584%, +422%); product time is close to the overall mean (+9%).|
|7|97|Administrative focus: administrative count is 95% above average; informational time is 95% below average and product time is 26% below.|

## A3.3 · Which partition is easier to interpret?
I would present **K = 4** as the main segmentation: light browsing, administrative browsing, information-intensive visits, and extensive product browsing. The largest group covers 332 of 500 sessions (66.4%). K = 7 adds useful detail: one group of 49 sessions concentrates on products without above-average activity elsewhere. But two groups contain only 4 and 10 sessions, so their averages describe very few observations. The decrease in inertia from 1,365.752 to 872.406 describes a tighter fit with more centers; it does not decide which segmentation is easier to communicate. These profiles describe visits in this sample, not permanent customer types.